## ESAT Input Perturbation Workflow

This notebook implements an input perturbation workflow for model evaluation.

#### Code Imports

In [ ]:
import time
import copy

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from tqdm.notebook import trange, tqdm, tnrange
from plotly.subplots import make_subplots
from esat.data.datahandler import DataHandler
from esat.model.sa import SA
from esat.model.batch_sa import BatchSA
from esat.data.analysis import ModelAnalysis, BatchAnalysis
from esat.error.bootstrap import Bootstrap
from esat_eval.simulator import Simulator
from esat_eval.factor_comparison import FactorCompare

pd.options.display.float_format = '{:.4f}'.format

In [ ]:
# Synethic dataset parameter value ranges
syn_factors_min = 3
syn_factors_max = 8

syn_features_min = 15
syn_features_max = 45

syn_samples_min = 200
syn_samples_max = 1000

outliers = True
outliers_p_min = 0.05
outliers_p_max = 0.1
outliers_mag_min = 1.1
outliers_mag_max = 2

noise_mean_min = 0.05
noise_mean_max = 0.15
noise_scale = 0.01

uncertainty_mean_min = 0.05
uncertainty_mean_max = 0.15
uncertainty_scale = 0.01

contr_curve_min_range = [0.0, 1.0]
contr_curve_max_range = [2.0, 5.0]
contr_curve_scale_range = [0.1, 0.5]

random_seed = 337
k_coef = 0.75

In [ ]:
rng = np.random.default_rng(seed=random_seed)

In [ ]:
# Initialize the simulator with the above parameters
def generate_synthetic_data(true_factor):
    n_features = rng.integers(low=syn_features_min, high=syn_features_max, size=1)[0]
    n_samples = rng.integers(low=syn_samples_min, high=syn_samples_max, size=1)[0]
    i_outlier_p = round(rng.uniform(low=outliers_p_min, high=outliers_p_max, size=1)[0], 2)
    i_outlier_mag = round(rng.uniform(low=outliers_mag_min, high=outliers_mag_max, size=1)[0], 2)
    contribution_max = round(rng.uniform(low=1.0, high=10.0, size=1)[0], 2)
    print(f"True Factors: {true_factor}, Features: {n_features}, Samples: {n_samples}, Outliers %: {i_outlier_p}, Outliers Magnitude: {i_outlier_mag}, Contribution Max: {contribution_max}")
    simulator = Simulator(seed=rng.integers(low=0, high=10, size=1)[0],
                          factors_n=true_factor,
                          features_n=n_features,
                          samples_n=n_samples,
                          outliers=outliers,
                          outlier_p=i_outlier_p,
                          outlier_mag=i_outlier_mag,
                          contribution_max=contribution_max,
                          noise_mean_min=noise_mean_min,
                          noise_mean_max=noise_mean_max,
                          noise_scale=noise_scale,
                          uncertainty_mean_min=uncertainty_mean_min,
                          uncertainty_mean_max=uncertainty_mean_max,
                          uncertainty_scale=uncertainty_scale,
                          verbose=False
                         )
    curved_factors_count = rng.integers(low=0, high=true_factor, size=1)[0]
    curved_factor_list = rng.choice(list(range(true_factor)), size=curved_factors_count, replace=False)
    for c_i in curved_factor_list:
        # parameters not used by the curve type are ignored
        i_curve_type = rng.choice(['uniform', 'decreasing', 'increasing', 'logistic', 'periodic'], size=1)[0]
        i_curve_min = rng.uniform(low=contr_curve_min_range[0], high=contr_curve_min_range[1], size=1)[0]
        i_curve_max = rng.uniform(low=contr_curve_max_range[0], high=contr_curve_max_range[1], size=1)[0]
        i_curve_scale = rng.uniform(low=contr_curve_scale_range[0], high=contr_curve_scale_range[1], size=1)[0]
        i_curve_frequency = rng.uniform(low=0.1, high=0.9, size=1)[0]
        
        # To keep all as uniform comment out the line below
        # simulator.update_contribution(factor_i=c_i, curve_type=i_curve_type, scale=i_curve_scale, frequency=i_curve_frequency, minimum=i_curve_min, maximum=i_curve_max)
    
    syn_input_df, syn_uncertainty_df = simulator.get_data()
    data_handler = DataHandler.load_dataframe(input_df=syn_input_df, uncertainty_df=syn_uncertainty_df)
    data_handler.metrics
    V, U = data_handler.get_data()
    return V, U

def perturb_input(v, _rng, perturb_p = 0.25, perturb_v = 0.05, normal:bool=True):
    i_V = copy.copy(v)
    if isinstance(perturb_p, float):
        perturb_p = [perturb_p for i in range(v.shape[1])]
    elif isinstance(perturb_p, list) and len(perturb_p) != v.shape[1]:
        perturb_p = [perturb_p[0] for i in range(v.shape[1])]
    if isinstance(perturb_v, float):
        perturb_v = [perturb_v for i in range(v.shape[1])]
    elif isinstance(perturb_v, list) and len(perturb_v) != v.shape[1]:
        perturb_v = [perturb_v[0] for i in range(v.shape[1])]
    for i, _p in enumerate(perturb_p):
        i_mask = _rng.random(size=v[:,i].shape) > _p
        if normal:
            i_mean = i_V[:,i]
            i_scale = 2*(i_V[:,i] * perturb_v[i])
            normal_iV = _rng.normal(loc=i_mean, scale=i_scale, size=i_mean.shape)
            ij_V = i_V[:,i]
            ij_V[i_mask] =  normal_iV[i_mask]
            ij_V[ij_V < 0.0] = 1e-12
            i_V[:,i] = ij_V
        else:
            v_matrix = _rng.uniform(low=0, high=perturb_v[i], size=v[:,i].shape)
            pn_matrix = _rng.random(size=v[:,i].shape)
            pn_matrix[pn_matrix > 0.5] = 1.0
            pn_matrix[pn_matrix <= 0.5] = -1.0
            
            ij_V = i_V[:,i]
            # for the cells where i_mask is True, the value of i_V is equal to i_V +/- v_matrix * i_V
            ij_V[i_mask] =  ij_V[i_mask] + (pn_matrix[i_mask] * v_matrix[i_mask] * ij_V[i_mask])
            ij_V[ij_V < 0.0] = 1e-12
            i_V[:,i] = ij_V
    return i_V


def run_perturbation(v, u, factors, random_seed, v_collection = None, perturb_p = 0.25, perturb_v = 0.05, sa_model=None, models=10, max_iter=10000, converge_n=50, converge_delta=0.01, threshold: float=0.9, pg_leave=True):
    # Runs a perturbation input batch instance
    # Steps:
    # 1. Create a SA instance using the provided iV, iU and true_k for the data and factor count (if one is not provided).
    # 2. Using a predefined % value (perturb_v) change and % instance change (perturb_p) for the dataset, or by feature (TO BE ADDED):
    #    a. Select perturb_p number of indecies from the input data matrix and change those values +/- up to perturb_v
    #    b. Use a provided collection of pre-perturbed v matrices
    # 3. With the perturbed dataset rerun the model using the base model H matrix.
    # 4. Repeast for n number of models
    # 5. Evaluate the results from all the perturb model profiles and concentrations, the ones that mapped (had a correlation above the threhsold) and provide the range of values for the factors.

    rng = np.random.default_rng(seed=random_seed)
    # step 1
    if sa_model is None:
        sa_model = SA(V=v, U=u, factors=factors, seed=random_seed, verbose=False)
        sa_model.initialize()
        sa_model.train(max_iter=max_iter, converge_delta=converge_delta, converge_n=converge_n)

    factor_mapping_p = []
    factor_coef_mean = []
    perturb_models = []
    p_i_V = v
    for i in tnrange(models, desc="Running Perturbations on base model", leave=pg_leave):
        if v_collection is None:
            i_V = perturb_input(v=v, perturb_p=perturb_p, perturb_v=perturb_v, _rng=rng)
        elif i > len(v_collection):
            i_V = perturb_input(v=v, perturb_p=perturb_p, perturb_v=perturb_v, _rng=rng)
        else:
            i_V = v_collection[i]
        i_sa_model = SA(i_V, U=u, factors=factors, seed=random_seed, verbose=False)
        i_sa_model.initialize(H=sa_model.H, W=sa_model.W)
        i_sa_model.train(max_iter=max_iter, converge_delta=converge_delta, converge_n=converge_n)
        perturb_models.append(copy.copy(i_sa_model))
        i_mapping = []
        coef_mapping = []
        for k in range(factors):
            base_factor = sa_model.H[k]
            k_factor = i_sa_model.H[k]
            base_k_correlation = FactorCompare.calculate_correlation(factor1=base_factor, factor2=k_factor)
            coef_mapping.append(base_k_correlation)
            if base_k_correlation >= threshold:
                i_mapping.append(1)
            else:
                i_mapping.append(0)
        factor_mapping_p.append(np.sum(i_mapping)/factors)
        factor_coef_mean.append(np.round(np.mean(coef_mapping), 4))
    results = {
        "k": factors,
        "seed": random_seed,
        "perturb %": perturb_p,
        "perturb v": perturb_v,
        "mean r2": factor_coef_mean,
        "mapping %": factor_mapping_p,
        "base_model": sa_model,
        "perturb_models": perturb_models,
    }
    return results
    

## Single Perturbation Instance

A single instance of the run_perturbation function is called. This will create a single SA instance to use as the base model with n_models being the number of perturbed instances to make (each independent of each other) for that single base model.

In [ ]:
%%time
true_k = 6
iV, iU = generate_synthetic_data(true_factor=true_k)

n_models = 10
threshold = 0.9

perturb_p = 0.5
perturb_v = 0.05

# The perturbed max percent change and percent occurrence can be defined by feature by providing a list of value, corresponding to a feature by index.
# perturb_v = [ rng.uniform(low=0.0, high=0.1, size=None) for i in range(iV.shape[1])]
# perturb_p = [ rng.uniform(low=0.1, high=0.5, size=None) for i in range(iV.shape[1])]

# If the input has already been perturbed, a set of perturbed Vs can be provided by passing in v_collection (a list of Vs that will be used in place of random perturbation).
# v_collection = None

batch_results0 = run_perturbation(factors=true_k, v=iV, u=iU, random_seed=random_seed, perturb_p=perturb_p, perturb_v=perturb_v, models=n_models, max_iter=20000, converge_n=50, converge_delta=0.01, threshold=threshold)

In [ ]:
base_H = batch_results0["base_model"].H
base_H = (base_H / np.sum(base_H, axis=0))

p1_H = batch_results0["perturb_models"][0].H

p_Hs = []
p_Qs = []
for p_model in batch_results0["perturb_models"]:
    p_H = (p_model.H / np.sum(p_model.H, axis=0))
    p_Hs.append(p_H)
    p_Qs.append(p_model.Qtrue)
perturb_H = np.dstack(p_Hs)
mean_perturb_H = np.mean(perturb_H, axis=2)
std_perturb_H = np.std(perturb_H, axis=2)
min_perturb_H = np.min(perturb_H, axis=2)
max_perturb_H = np.max(perturb_H, axis=2)

feature_labels = [f"Feature {i+1}" for i in range(base_H.shape[1])]
print(f"QTrue - Base: {np.round(batch_results0["base_model"].Qtrue,4)}, Perturb Mean: {np.round(np.mean(p_Qs),4)}, Perturb STD: {np.round(np.std(p_Qs),4)}, Perturb Min: {np.round(np.min(p_Qs),4)}, Perturb Max: {np.round(np.max(p_Qs),4)}")
print(f"% Profiles Mapped: {np.round(100*np.mean(batch_results0["mapping %"]), 2)}, Mean R2: {np.round(np.mean(batch_results0['mean r2']), 4)}")

In [ ]:
factor_i = 1
f1_dict = {
    "Base": base_H[factor_i],
    "Mean Perturb": mean_perturb_H[factor_i],
    "% diff": np.round(100*(mean_perturb_H[factor_i] - base_H[factor_i])/((mean_perturb_H[factor_i] + base_H[factor_i])/2) , 4),
    "STD Perturb": std_perturb_H[factor_i],
    "Min Perturb": min_perturb_H[factor_i],
    "Max Perturb": max_perturb_H[factor_i],
}
f1_dict["% diff"][f1_dict["Mean Perturb"] < 1e-6] = 0.0 

f1_df = pd.DataFrame(f1_dict, index=feature_labels).round(10)
f1_df

In [ ]:
factor_i = 0
factor_p0_fig = make_subplots()
for i in range(len(feature_labels)):
    f_feature_i = perturb_H[factor_i,i]
    factor_p0_fig.add_trace(go.Box(y=f_feature_i, name=feature_labels[i]))
factor_p0_fig.add_trace(go.Scatter(x=feature_labels, y=base_H[factor_i], name="Base", mode="markers", marker_color="black"))
factor_p0_fig.update_layout(title=f"Perturbed Factor {factor_i} Profile Results", width=1200, height=800, hovermode='x unified')
factor_p0_fig.update_yaxes(title_text="Normalized Profile")
factor_p0_fig.show()

## Batch Perturbation Instance

Run multiple single perturbation instances, creating a collection of base models and their corresponding perturbed models. Create a dictionary of factor profiles, linking them to a model and factor profile by correlation mapping (above a specified threshold).

Aggregated results will include:
1. A list of factor profiles and their % occurrence across all base models
2. The base models which include the highest % occuring factor profiles
3. The factor profile ranges across the perturbed models sharing those factor profiles.

In [ ]:
def run_batch_perturbation(factors, v, u, random_seed, v_collection=None, batches=10, perturb_p=0.5, perturb_v=0.05, models=100, max_iter=20000, converge_n=20, converge_delta=0.01, threshold=0.9):
    batch_collections = []
    for b in trange(batches, desc='Running Perturbation Instances'):
        b_seed = rng.integers(low=1, high=1e10)
        batch_results = run_perturbation(factors=factors, v=v, u=u, random_seed=b_seed, v_collection=v_collection, perturb_p=perturb_p, perturb_v=perturb_v, models=n_models, max_iter=max_iter, converge_n=converge_n, converge_delta=converge_delta, threshold=threshold, pg_leave=False)
        batch_collections.append(batch_results)
    return batch_collections

def check_factor_catalog(factor_profile, factor_catalog):
    best_r2 = 0.0
    mapped_k = -1
    for factor_i, factor_details in factor_catalog.items():
        fk_r2_list = []
        for f_profiles in factor_details["profile"]:
            fk_r2 = FactorCompare.calculate_correlation(f_profiles, factor_profile)
            fk_r2_list.append(fk_r2)
        fk_r2_mean = np.mean(fk_r2_list)
        if fk_r2_mean > best_r2:
            best_r2 = fk_r2_mean
            mapped_k = factor_details["idx"]
    return best_r2, mapped_k

def map_batch_factors(batch_results):
    base_models = [bm["base_model"] for bm in batch_results]
    factor_n = 0
    factor_catalog = {}
    for batch_i, batch_result in enumerate(base_models):
        H_i = batch_result.H
        norm_Hi = (H_i / np.sum(H_i, axis=0))
        for k in range(H_i.shape[0]):
            if batch_i == 0:
                factor_catalog[f"Factor {factor_n}"] = {"profile": [norm_Hi[k]], "models": [batch_i], "r2": [1.0], "mapping": [k], "idx": factor_n}
                factor_n += 1
            else:
                best_r2, mapped_k = check_factor_catalog(norm_Hi[k], factor_catalog)
                if best_r2 > threshold:
                    factor_catalog[f"Factor {mapped_k}"]["profile"].append(norm_Hi[k])
                    factor_catalog[f"Factor {mapped_k}"]["models"].append(batch_i)
                    factor_catalog[f"Factor {mapped_k}"]["r2"].append(best_r2)
                    factor_catalog[f"Factor {mapped_k}"]["mapping"].append(k)
                else:
                    factor_catalog[f"Factor {factor_n}"] = {"profile": [norm_Hi[k]], "models": [batch_i], "r2": [1.0], "mapping": [k], "idx": factor_n}
                    factor_n += 1
    return factor_catalog

In [ ]:
%%time
n_factors = 6
n_batches = 20
n_models = 5
threshold = 0.95

perturb_p = 0.5
perturb_v = 0.05

# The perturbed max percent change and percent occurrence can be defined by feature by providing a list of value, corresponding to a feature by index.
# perturb_v = [ rng.uniform(low=0.0, high=0.1, size=None) for i in range(iV.shape[1])]
# perturb_p = [ rng.uniform(low=0.1, high=0.5, size=None) for i in range(iV.shape[1])]

batch_results = run_batch_perturbation(factors=n_factors, v=iV, u=iU, batches=n_batches, random_seed=random_seed, perturb_p=perturb_p, perturb_v=perturb_v, models=n_models, max_iter=10000, converge_n=20, converge_delta=0.01, threshold=threshold)

In [ ]:
# Calculate the occurrences of each factor in all the batch base models

profiles_dict = map_batch_factors(batch_results=batch_results)

In [ ]:
# Display factor counts, occurrences, and mean r2 value
factor_occurrence = [round(100 * len(f["models"])/n_batches, 2) for _, f in profiles_dict.items()]
factor_count = [len(f["models"]) for _, f in profiles_dict.items()]
factor_mean_r2 = [round(np.mean(f["r2"]), 4) for _, f in profiles_dict.items()]
profiles_df = pd.DataFrame(data={"factors": profiles_dict.keys(), "count": factor_count,  "% occurrence": factor_occurrence, "mean R2": factor_mean_r2})
profiles_df

In [ ]:
# Calculate scores for each of the models based upon the occurrences of the profiles across all batches
model_scores = [0 for i in range(n_batches)]
for f_name, factor_details in profiles_dict.items():
    f_i = int(f_name.split(" ")[1])
    for i in factor_details["models"]:
        model_scores[i] += factor_occurrence[f_i]
models_ranked = np.flip(np.argsort(model_scores))
print(f"Model Scores by Index: {model_scores}")
print(f"Models Ordered by Score: {models_ranked}")

In [ ]:
# Calcualte a specific factor profile range across all perturbations and models
# For the specified factor, calculate the mean profile values from all the base models.
# Then stack all perturbed models for that factor and generate box plot
factor_selected = 1
f_name = f"Factor {factor_selected}"
f_details = profiles_dict[f_name]

f_matrix = np.array(f_details["profile"])
f_matrix = np.dstack(f_matrix)[0]

# Factor ranges from the base model profiles
factor_base_fig = make_subplots()
for i in range(len(feature_labels)):
    b_feature_i = f_matrix[i]
    factor_base_fig.add_trace(go.Box(y=b_feature_i, name=feature_labels[i]))

factor_base_fig.update_layout(title=f"Base Factor {factor_selected} Profile Results - N Models: {len(f_details["models"])}", width=1200, height=800, hovermode='x unified')
factor_base_fig.update_yaxes(title_text="Normalized Profile", range=[0, 1.0])
factor_base_fig.show()

# Factor profile ranges from the perturbed models which have a base model that mapped
perturb_profile = []
for i, base_i in enumerate(f_details["models"]):
    batch_result = batch_results[base_i]["perturb_models"]
    mapped_factor = f_details['mapping'][i]
    for p_model in batch_result:
        norm_H = p_model.H / np.sum(p_model.H, axis=0)
        perturb_profile.append(norm_H[mapped_factor])
perturb_matrix = np.array(perturb_profile)
perturb_matrix = np.dstack(perturb_matrix)[0]

factor_p_fig = make_subplots()

for i in range(len(feature_labels)):
    b_feature_i = perturb_matrix[i]
    factor_p_fig.add_trace(go.Box(y=b_feature_i, name=feature_labels[i])) 
factor_p_fig.add_trace(go.Scatter(x=feature_labels, y=np.mean(f_matrix, axis=1), name="Base Mean", mode="markers", marker_color="black"))
factor_p_fig.update_layout(title=f"Perturbed Factor {factor_selected} Profile Results", width=1200, height=800, hovermode='x unified')
factor_p_fig.update_yaxes(title_text="Normalized Profile", range=[0, 1.0])
factor_p_fig.show()

# Factor line plot from factor catelog
factor_mapping_fig = make_subplots()
for f_i in range(len(f_details["profile"])):
    factor_mapping_fig.add_trace(go.Scatter(y=f_details["profile"][f_i], x=feature_labels, name=f"Model {f_details["models"][f_i]} - Factor {f_details["mapping"][f_i]} - R2 {np.round(f_details["r2"][f_i],4)}"))

factor_mapping_fig.update_layout(title=f"{f_name} Mapping", width=1200, height=800, hovermode='x unified')
factor_mapping_fig.update_yaxes(title_text="Normalized Profile", range=[0, 1.0])
factor_mapping_fig.show()

In [ ]:
# When running a large number of initial models use the mean values of the most frequent profiles as the initial profiles for a new SA model and see how well it performs, in terms of loss.

In [ ]:
profiles_df_sorted = profiles_df.sort_values(by='% occurrence', ascending=False)
profiles_df_sorted

In [ ]:
optimal_factor_list = list(profiles_df_sorted["factors"][0:5])
optimal_factor_list

In [ ]:
# optimal_factor_list = ['Factor 4']

In [ ]:
case1_H = []
case2_H = []
for o_factor in optimal_factor_list:
    factor_range = len(profiles_dict[o_factor]["profile"])
    # randomly select a profile from the cataloged factor profile
    factor_idx = rng.integers(low=0, high=factor_range, size=None)
    i_factor = profiles_dict[o_factor]["profile"][factor_idx]
    i_norm_factor = i_factor / np.sum(i_factor)
    case1_H.append(i_norm_factor)
    case2_H.append(i_factor)
case1_H = np.array(case1_H)
case2_H = np.array(case2_H)
case3_H = case1_H / np.sum(case1_H, axis=0)

In [ ]:
def compile_optimal_tests():
    
    opt_norm_H1 = optimal_sa1.H / np.sum(optimal_sa1.H, axis=0)
    opt_norm_H2 = optimal_sa2.H / np.sum(optimal_sa2.H, axis=0)
    opt_norm_H3 = optimal_sa3.H / np.sum(optimal_sa3.H, axis=0)
    opt_norm_H4 = optimal_sa4.H / np.sum(optimal_sa4.H, axis=0)
    
    case_1_results = {}
    case_2_results = {}
    case_3_results = {}
    case_4_results = {}
    
    df_data = []
    
    for f_i in range(n_factors):
        best_r20, mapped_k0 = check_factor_catalog(factor_profile=opt_norm_H1[f_i], factor_catalog=profiles_dict)
        best_r21, mapped_k1 = check_factor_catalog(factor_profile=opt_norm_H2[f_i], factor_catalog=profiles_dict)
        best_r22, mapped_k2 = check_factor_catalog(factor_profile=opt_norm_H3[f_i], factor_catalog=profiles_dict)
        best_r24, mapped_k4 = check_factor_catalog(factor_profile=opt_norm_H4[f_i], factor_catalog=profiles_dict)
    
        if f_i < len(optimal_factor_list):
            original_factor = int(optimal_factor_list[f_i].split(" ")[1])
            original_r2_0 = FactorCompare.calculate_correlation(case1_H[f_i], opt_norm_H1[f_i])
            original_r2_1 = FactorCompare.calculate_correlation(case2_H[f_i], opt_norm_H2[f_i])
            original_r2_2 = FactorCompare.calculate_correlation(case3_H[f_i], opt_norm_H3[f_i])
            original_r2_4 = FactorCompare.calculate_correlation(case2_H[f_i], opt_norm_H4[f_i])
        else:
            original_factor = "NA"
            original_r2_0 = -1
            original_r2_1 = -1
            original_r2_2 = -1
            original_r2_4 = -1
        
        case_1_results[f_i] = {"profile": opt_norm_H1[f_i], "map": mapped_k0, "r2": best_r20}
        case_2_results[f_i] = {"profile": opt_norm_H2[f_i], "map": mapped_k1, "r2": best_r21}
        case_3_results[f_i] = {"profile": opt_norm_H3[f_i], "map": mapped_k2, "r2": best_r22}
        case_4_results[f_i] = {"profile": opt_norm_H4[f_i], "map": mapped_k4, "r2": best_r24}
    
        df_data.append({
            "Factor": f_i,
            "Original": original_factor,
            "Map C1": mapped_k0,
            "Map C2": mapped_k1,
            "Map C3": mapped_k2,
            "Map C4": mapped_k4,
            "Original 1 R2": original_r2_0,
            "Original 2 R2": original_r2_1,
            "Original 3 R2": original_r2_2,
            "Original 4 R2": original_r2_4,
            "Mapped 1 R2": best_r20,
            "Mapped 2 R2": best_r21,
            "Mapped 3 R2": best_r22,
            "Mapped 4 R2": best_r24,
        })
    optimal_df = pd.DataFrame(data=df_data)
    return optimal_df

In [ ]:
base_QTrues = [bm["base_model"].Qtrue for bm in batch_results]
base_QRobust = [bm["base_model"].Qrobust for bm in batch_results]
best_model = batch_results[models_ranked[0]]["base_model"]
print(f"Base Model - Mean QTrue: {np.round(np.mean(base_QTrues), 4)}, Min QTrue: {np.round(np.min(base_QTrues), 4)}, Max QTrue: {np.round(np.max(base_QTrues), 4)}")
print(f"Base Model - Mean QRobust: {np.round(np.mean(base_QRobust), 4)}, Min QRobust: {np.round(np.min(base_QRobust), 4)}, Max QRobust: {np.round(np.max(base_QRobust), 4)}")
print(f"Highest Occurrence Model - QTrue: {np.round(best_model.Qtrue, 4)}, QRobust: {np.round(best_model.Qrobust, 4)}")

In [ ]:
test_seed = rng.integers(low=1, high=1e10)

# Doubly normalized factors
optimal_sa1 = SA(V=iV, U=iU, factors=6, seed=test_seed, verbose=True)
optimal_sa1.initialize(H=case1_H)
optimal_sa1.train(max_iter=20000, converge_delta=0.01, converge_n=20)

# Original normalized factors
optimal_sa2 = SA(V=iV, U=iU, factors=6, seed=test_seed, verbose=True)
optimal_sa2.initialize(H=case2_H)
optimal_sa2.train(max_iter=20000, converge_delta=0.01, converge_n=20)

# Doubly normalized factors are normalized to the optimal H
optimal_sa3 = SA(V=iV, U=iU, factors=6, seed=test_seed, verbose=True)
optimal_sa3.initialize(H=case3_H)
optimal_sa3.train(max_iter=20000, converge_delta=0.01, converge_n=20)

optimal_sa4 = SA(V=iV, U=iU, factors=6, seed=test_seed, verbose=True)
optimal_sa4.initialize()
for i in range(len(case2_H)):
    optimal_sa4.H[i] = case2_H[i]
optimal_sa4.H = optimal_sa4.H / np.sum(optimal_sa4.H, axis=0)
optimal_sa4.H[optimal_sa4.H <= 0.0] = 1e-12
optimal_sa4.train(max_iter=20000, converge_delta=0.01, converge_n=20)

print("\nCase 1: Doubly Normalized, Case 2: Original Normalization, Case 3: Renormalized to selected profiles, Case 4: Manually normalized\n")

optimal_df = compile_optimal_tests()
optimal_df

In [ ]:
# Reference Model
ref_sa = SA(V=iV, U=iU, factors=6, seed=random_seed, verbose=True)
ref_sa.initialize()
ref_sa.train(max_iter=20000, converge_delta=0.01, converge_n=20)

In [ ]:
test_i = 0
Hi = [ref_sa.H[test_i]]
Wi = ref_sa.W[:,test_i]
Wi = np.reshape(Wi, (Wi.shape[0], 1))
i_v = np.matmul(Wi, Hi)
i_v.shape

In [ ]:
ref_norm_H = ref_sa.H / np.sum(ref_sa.H, axis=0)
for f_i in range(n_factors):
    best_r20, mapped_k0 = check_factor_catalog(factor_profile=ref_norm_H[f_i], factor_catalog=profiles_dict)
    print(f"Reference SA Factor {f_i} - Mapped K: {mapped_k0}, best R2: {best_r20}")

In [ ]:
for factor_selected in range(n_factors):
    
    mapped_factor = case_1_results[factor_selected]
    
    f_name = f"Factor {mapped_factor['map']}"
    f_details = profiles_dict[f_name]
    
    f_matrix = np.array(f_details["profile"])
    f_matrix = np.dstack(f_matrix)[0]
    
    # Factor ranges from the base model profiles
    factor_base_fig = make_subplots()
    for i in range(len(feature_labels)):
        b_feature_i = f_matrix[i]
        factor_base_fig.add_trace(go.Box(y=b_feature_i, name=feature_labels[i]))
    
    factor_base_fig.add_trace(go.Scatter(x=feature_labels, y=mapped_factor["profile"], name=f"Optimal Factor ({factor_selected})", mode="lines+markers", marker_color="black"))
    factor_base_fig.add_trace(go.Scatter(x=feature_labels, y=optimal_H[factor_selected], name="Original Value", mode="lines+markers", marker_color="brown"))
    
    factor_base_fig.update_layout(title=f"Optimal Profile Results: {f_name} - Original: {optimal_factor_list[factor_selected]} - R2: {np.round(mapped_factor["r2"], 4)}", width=1200, height=800, hovermode='x unified')
    factor_base_fig.update_yaxes(title_text="Normalized Profile", range=[0, 1.0])
    factor_base_fig.show()
